In [1]:
import sympy as sp
from scipy.special import xlogy as scipy_xlogy

class _CustomXLogXSymPy(sp.Function):
    nargs = 1 # Specifies that this function takes one argument.
    # No other methods are strictly needed if it's just a placeholder
    # for lambdify's modules argument.

In [2]:
def preprocess_sympy_expr_for_xlogx(expr, expand_terms=True):
    """
    Pre-processes a SymPy expression:
    1. Optionally expands it.
    2. Replaces occurrences of 'variable * log(variable)' with _CustomXLogXSymPy(variable).
    """
    if expand_terms:
        processed_expr = sp.expand(expr)
        processed_expr = sp.expand_log(processed_expr, force=True)
    else:
        processed_expr = expr

    v_sym = sp.Wild('_v_wild_xlogx', instanceof=sp.Symbol)
    
    pattern1 = v_sym * sp.log(v_sym)
    pattern2 = sp.log(v_sym) * v_sym # Commutative case
    
    replaced_expr = processed_expr.replace(
        pattern1,
        lambda actual_matched_variable: _CustomXLogXSymPy(actual_matched_variable)
    )
    replaced_expr = replaced_expr.replace(
        pattern2,
        lambda actual_matched_variable: _CustomXLogXSymPy(actual_matched_variable)
    )
    
    return replaced_expr

In [3]:
x, y = sp.symbols('x y')
original_expr = x * (sp.log(x) + y) + sp.log(x**x) + y * sp.log(y)
print(f"Original SymPy expression: {original_expr}")

Original SymPy expression: x*(y + log(x)) + y*log(y) + log(x**x)


In [22]:
processed_expr = sp.expand(original_expr)
processed_expr = sp.expand_log(processed_expr, force=True)
print(processed_expr)
v_sym = sp.Wild('_v_wild_xlogx', instanceof=sp.Symbol)
    
pattern1 = v_sym * sp.log(v_sym)
pattern2 = sp.log(v_sym) * v_sym # Commutative case
replaced_expr = processed_expr.subs(
        pattern1,
        y
    )
print(replaced_expr)

x*y + 2*x*log(x) + y*log(y)
x*y + 2*x*log(x) + y*log(y)


In [4]:
# 1. Pre-process the expression
processed_expr = preprocess_sympy_expr_for_xlogx(original_expr)
print(f"Processed SymPy expression: {processed_expr}")
# Expected: _CustomXLogXSymPy(x) + x*y + _CustomXLogXSymPy(x) + _CustomXLogXSymPy(y)

TypeError: preprocess_sympy_expr_for_xlogx.<locals>.<lambda>() got an unexpected keyword argument '_v_wild_xlogx'